# exp005: Phase 2+3 - Pseudo Label & Retrain

- **Phase 2**: Phase 1 モデルで train_soundscapes に疑似ラベル（soft label）生成
- **Phase 3**: train_audio (hard label) + train_soundscapes (soft label) で再学習

**Input**: `birdclef-2026`, `exp005` (Phase 1 weights)
**Output**: `weights/best_fold0.pth` (再学習済み重み)

In [ ]:
!pip install -q timm torchaudio scikit-learn

In [ ]:
import os, glob, pathlib, random, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import timm
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}, GPUs: {torch.cuda.device_count()}')

In [ ]:
# ── Config ────────────────────────────────────────────────
CFG = dict(
    # Audio
    sample_rate      = 32000,
    n_mels           = 128,
    n_fft            = 1024,
    hop_length       = 320,
    fmin             = 20,
    fmax             = 16000,
    segment_sec      = 5,
    # Model
    model_name       = 'tf_efficientnetv2_s',
    num_classes      = 234,
    # Phase 2
    pseudo_batch_size = 32,
    # Phase 3
    seed             = 42,
    n_folds          = 5,
    train_fold       = 0,
    epochs           = 20,
    batch_size       = 32,           # increased (5s input uses less memory)
    num_workers      = 2,
    lr               = 5e-4,
    weight_decay     = 1e-4,
    warmup_epochs    = 1,
    use_amp          = True,
    samples_per_epoch = 60000,
    early_stopping   = 5,            # stop if no improvement for N epochs
    # SpecAugment
    spec_aug         = True,
    freq_mask_param  = 27,
    time_mask_param  = 100,
    # Gaussian Noise (train_audio only)
    noise_std        = 0.005,
    noise_prob       = 0.5,
)

SEG_SAMPLES = CFG['segment_sec'] * CFG['sample_rate']  # 160000

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG['seed'])

In [ ]:
# ── Paths ─────────────────────────────────────────────────
_s = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
COMP_DIR = os.path.dirname(_s[0]) if _s else '/kaggle/input/birdclef-2026'

TRAIN_CSV       = f'{COMP_DIR}/train.csv'
SAMPLE_SUB_CSV  = f'{COMP_DIR}/sample_submission.csv'
TRAIN_AUDIO_DIR = f'{COMP_DIR}/train_audio'
SOUNDSCAPE_DIR  = f'{COMP_DIR}/train_soundscapes'

WEIGHT_PATH = '/kaggle/input/datasets/maekeso/exp005/weights/best_fold0.pth'

WEIGHT_DIR = '/kaggle/working/weights'
LOG_DIR    = '/kaggle/working/logs'
os.makedirs(WEIGHT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

for label, path in [('TRAIN_AUDIO_DIR', TRAIN_AUDIO_DIR), ('SOUNDSCAPE_DIR', SOUNDSCAPE_DIR), ('WEIGHT_PATH', WEIGHT_PATH)]:
    print(f'  {"OK" if os.path.exists(path) else "NG"} {label}: {path}')

# Labels
sub_df = pd.read_csv(SAMPLE_SUB_CSV, nrows=0)
LABELS = [c for c in sub_df.columns if c != 'row_id']
LABEL2IDX = {l: i for i, l in enumerate(LABELS)}
print(f'Species: {len(LABELS)}')

In [ ]:
# ── Model Definition ──────────────────────────────────────
class AttBlockV2(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.att = nn.Conv1d(in_features, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(in_features, num_classes, kernel_size=1, bias=True)
        nn.init.xavier_uniform_(self.att.weight)
        nn.init.xavier_uniform_(self.cla.weight)
        nn.init.constant_(self.att.bias, 0)
        nn.init.constant_(self.cla.bias, 0)

    def forward(self, x):
        att = torch.softmax(torch.tanh(self.att(x)), dim=-1)
        cla = self.cla(x)
        return (att * cla).sum(dim=-1)

class BirdCLEFSED(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(
            CFG['model_name'], pretrained=pretrained,
            in_chans=1, num_classes=0, global_pool='',
        )
        in_features = self.backbone.num_features
        self.bn = nn.BatchNorm1d(in_features)
        self.dropout = nn.Dropout(p=0.3)
        self.att_block = AttBlockV2(in_features, CFG['num_classes'])

    def forward(self, x):
        feat = self.backbone.forward_features(x)
        feat = feat.mean(dim=2)
        feat = self.bn(feat)
        feat = self.dropout(feat)
        return self.att_block(feat)

In [ ]:
# ── Mel Transform & SpecAugment ──────────────────────────
mel_transform = nn.Sequential(
    T.MelSpectrogram(
        sample_rate=CFG['sample_rate'], n_fft=CFG['n_fft'],
        hop_length=CFG['hop_length'], n_mels=CFG['n_mels'],
        f_min=CFG['fmin'], f_max=CFG['fmax'],
    ),
    T.AmplitudeToDB(top_db=80),
).to(DEVICE)

freq_masking = T.FrequencyMasking(freq_mask_param=CFG['freq_mask_param']).to(DEVICE)
time_masking = T.TimeMasking(time_mask_param=CFG['time_mask_param']).to(DEVICE)

def waveform_to_spec(waveforms, training=False):
    with torch.no_grad():
        specs = mel_transform(waveforms)
    specs = specs - specs.amin(dim=(-2, -1), keepdim=True)
    specs = specs / (specs.amax(dim=(-2, -1), keepdim=True) + 1e-8)
    if training and CFG['spec_aug']:
        specs = freq_masking(specs)
        specs = time_masking(specs)
    return specs.unsqueeze(1)

In [ ]:
# ── Load Phase 1 Model ────────────────────────────────────
model = BirdCLEFSED(pretrained=False).to(DEVICE)
ckpt = torch.load(WEIGHT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Phase 1 weights loaded: epoch {ckpt["epoch"]}, AUC {ckpt["best_auc"]:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════
# Phase 2: Pseudo Label Generation on train_soundscapes
# ══════════════════════════════════════════════════════════

soundscape_files = sorted(glob.glob(f'{SOUNDSCAPE_DIR}/*.ogg'))
print(f'Soundscape files: {len(soundscape_files)}')

@torch.no_grad()
def generate_pseudo_labels(filepath):
    """Generate soft labels for each 5s segment in a soundscape file."""
    waveform, sr = torchaudio.load(filepath)
    if sr != CFG['sample_rate']:
        waveform = torchaudio.functional.resample(waveform, sr, CFG['sample_rate'])
    audio = waveform.mean(dim=0)
    total_samples = len(audio)
    
    segments, end_secs = [], []
    for end_sample in range(SEG_SAMPLES, total_samples + 1, SEG_SAMPLES):
        start = end_sample - SEG_SAMPLES
        chunk = audio[start:end_sample]  # exactly 5s
        if len(chunk) < SEG_SAMPLES:
            chunk = F.pad(chunk, (0, SEG_SAMPLES - len(chunk)))
        segments.append(chunk)
        end_secs.append(end_sample // CFG['sample_rate'])
    
    if not segments:
        return [], []
    
    batch = torch.stack(segments).to(DEVICE)
    all_probs = []
    for i in range(0, len(batch), CFG['pseudo_batch_size']):
        b = batch[i:i+CFG['pseudo_batch_size']]
        specs = waveform_to_spec(b)
        with autocast(enabled=CFG['use_amp']):
            logits = model(specs)
        all_probs.append(torch.sigmoid(logits.float()).cpu().numpy())
    
    return end_secs, np.concatenate(all_probs, axis=0)

# Run pseudo labeling
pseudo_rows = []
t0 = time.time()

for fi, fpath in enumerate(tqdm(soundscape_files, desc='Phase 2')):
    fname = pathlib.Path(fpath).stem
    try:
        end_secs, probs = generate_pseudo_labels(fpath)
    except Exception as e:
        print(f'  Error: {fname}: {e}')
        continue
    for es, p in zip(end_secs, probs):
        row = {'filepath': fpath, 'end_sec': es}
        for i, label in enumerate(LABELS):
            row[label] = float(p[i])
        pseudo_rows.append(row)

pseudo_df = pd.DataFrame(pseudo_rows)
elapsed = time.time() - t0
print(f'Phase 2 done: {len(pseudo_df)} segments from {len(soundscape_files)} files in {elapsed:.0f}s')

In [ ]:
# ── Pseudo Label Statistics ───────────────────────────────
probs_matrix = pseudo_df[LABELS].values
print(f'Shape: {probs_matrix.shape}')
print(f'Mean: {probs_matrix.mean():.6f}, Max: {probs_matrix.max():.4f}')
for th in [0.5, 0.3, 0.1]:
    n = (probs_matrix > th).sum()
    n_rows = (probs_matrix > th).any(axis=1).sum()
    print(f'  Threshold {th}: {n} positives in {n_rows} segments')

# Save pseudo labels
pseudo_df.to_csv(f'{LOG_DIR}/pseudo_labels.csv', index=False)
print(f'Pseudo labels saved to {LOG_DIR}/pseudo_labels.csv')

In [ ]:
# ══════════════════════════════════════════════════════════
# Phase 3: Retrain with train_audio + train_soundscapes
# ══════════════════════════════════════════════════════════

# ── Datasets ──────────────────────────────────────────────
class TrainAudioDataset(Dataset):
    """train_audio with one-hot hard labels."""
    def __init__(self, df, mode='train'):
        self.df = df.reset_index(drop=True)
        self.mode = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio = self._load_audio(row['filename'])
        # One-hot label
        label = torch.zeros(CFG['num_classes'])
        pidx = LABEL2IDX.get(str(row['primary_label']), -1)
        if pidx >= 0:
            label[pidx] = 1.0
        return audio, label

    def _load_audio(self, filename):
        path = f'{TRAIN_AUDIO_DIR}/{filename}'
        try:
            waveform, sr = torchaudio.load(path)
        except Exception:
            return torch.zeros(SEG_SAMPLES)
        if sr != CFG['sample_rate']:
            waveform = torchaudio.functional.resample(waveform, sr, CFG['sample_rate'])
        audio = waveform.mean(dim=0)
        if len(audio) >= SEG_SAMPLES:
            start = random.randint(0, len(audio) - SEG_SAMPLES) if self.mode == 'train' else (len(audio) - SEG_SAMPLES) // 2
            audio = audio[start:start + SEG_SAMPLES]
        else:
            audio = F.pad(audio, (0, SEG_SAMPLES - len(audio)))
        if self.mode == 'train' and random.random() < CFG['noise_prob']:
            audio = audio + torch.randn_like(audio) * CFG['noise_std']
        return audio


class SoundscapeDataset(Dataset):
    """train_soundscapes with soft pseudo labels."""
    def __init__(self, pseudo_df):
        self.pseudo_df = pseudo_df.reset_index(drop=True)
        self.label_values = pseudo_df[LABELS].values.astype(np.float32)

    def __len__(self):
        return len(self.pseudo_df)

    def __getitem__(self, idx):
        row = self.pseudo_df.iloc[idx]
        audio = self._load_segment(row['filepath'], int(row['end_sec']))
        label = torch.from_numpy(self.label_values[idx])
        return audio, label

    def _load_segment(self, filepath, end_sec):
        sr = CFG['sample_rate']
        end_sample = end_sec * sr
        start_sample = end_sample - SEG_SAMPLES
        num_frames = SEG_SAMPLES
        try:
            waveform, file_sr = torchaudio.load(
                filepath, frame_offset=start_sample, num_frames=num_frames
            )
            if file_sr != sr:
                waveform = torchaudio.functional.resample(waveform, file_sr, sr)
            audio = waveform.mean(dim=0)
        except Exception:
            return torch.zeros(SEG_SAMPLES)
        if len(audio) < SEG_SAMPLES:
            audio = F.pad(audio, (0, SEG_SAMPLES - len(audio)))
        return audio

print('Dataset classes defined')

In [ ]:
# ── Data Split & DataLoader ───────────────────────────────
train_df = pd.read_csv(TRAIN_CSV)
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df['primary_label'])):
    train_df.loc[val_idx, 'fold'] = fold

FOLD = CFG['train_fold']
tr_df = train_df[train_df['fold'] != FOLD].reset_index(drop=True)
va_df = train_df[train_df['fold'] == FOLD].reset_index(drop=True)

audio_ds = TrainAudioDataset(tr_df, mode='train')
soundscape_ds = SoundscapeDataset(pseudo_df)
combined_ds = ConcatDataset([audio_ds, soundscape_ds])

# Balanced sampling: equal weight for audio vs soundscape
n_audio = len(audio_ds)
n_sc = len(soundscape_ds)
w_audio = 1.0 / n_audio
w_sc = 1.0 / n_sc
sample_weights = [w_audio] * n_audio + [w_sc] * n_sc
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=CFG['samples_per_epoch'],
    replacement=True,
)

tr_loader = DataLoader(
    combined_ds, batch_size=CFG['batch_size'],
    sampler=sampler, num_workers=CFG['num_workers'],
    pin_memory=True, drop_last=True,
)
va_loader = DataLoader(
    TrainAudioDataset(va_df, mode='valid'),
    batch_size=CFG['batch_size'] * 2, shuffle=False,
    num_workers=CFG['num_workers'], pin_memory=True,
)

print(f'Train: {n_audio} audio + {n_sc} soundscape = {len(combined_ds)} total')
print(f'  Sampled per epoch: {CFG["samples_per_epoch"]} (~50/50 balanced)')
print(f'  Batches per epoch: {len(tr_loader)}')
print(f'Valid: {len(va_df)} (train_audio fold {FOLD})')

In [ ]:
# ── Utilities ─────────────────────────────────────────────
class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = self.avg = self.sum = self.count = 0.0
    def update(self, val, n=1):
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def compute_roc_auc(targets, preds):
    aucs = []
    for i in range(targets.shape[1]):
        if targets[:, i].sum() == 0:
            continue
        try:
            aucs.append(roc_auc_score(targets[:, i], preds[:, i]))
        except Exception:
            pass
    return float(np.mean(aucs)) if aucs else 0.0

In [ ]:
# ── Training & Validation Functions ───────────────────────
criterion = nn.BCEWithLogitsLoss()

def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    loss_meter = AverageMeter()
    for waveforms, targets in tqdm(loader, desc='  train', leave=False):
        waveforms = waveforms.to(DEVICE)
        targets = targets.to(DEVICE)
        optimizer.zero_grad()
        specs = waveform_to_spec(waveforms, training=True)
        with autocast(enabled=CFG['use_amp']):
            logits = model(specs)
            loss = criterion(logits, targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        loss_meter.update(loss.item(), waveforms.size(0))
    return loss_meter.avg

@torch.no_grad()
def validate(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    for waveforms, targets in tqdm(loader, desc='  valid', leave=False):
        waveforms = waveforms.to(DEVICE)
        specs = waveform_to_spec(waveforms, training=False)
        with autocast(enabled=CFG['use_amp']):
            logits = model(specs)
        all_preds.append(torch.sigmoid(logits.float()).cpu().numpy())
        all_targets.append(targets.numpy())
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    auc = compute_roc_auc(all_targets, all_preds)
    return auc

In [ ]:
# ── Phase 3: Training ─────────────────────────────────────
# Re-init from Phase 1 weights (already loaded)
optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-6)
scaler = GradScaler(enabled=CFG['use_amp'])

WEIGHT_PATH_OUT = f'{WEIGHT_DIR}/best_fold{FOLD}.pth'
best_auc, best_epoch = 0.0, 0
log_rows = []

print('=' * 60)
print(f'Phase 3 | Fold {FOLD} | {CFG["epochs"]} epochs | BCE loss')
print(f'  LR: {CFG["lr"]} | samples/epoch: {CFG["samples_per_epoch"]}')
print(f'  Early stopping: {CFG["early_stopping"]} epochs')
print('=' * 60)

for epoch in range(1, CFG['epochs'] + 1):
    t_start = time.time()
    if epoch <= CFG['warmup_epochs']:
        for pg in optimizer.param_groups:
            pg['lr'] = CFG['lr'] * epoch / CFG['warmup_epochs']

    tr_loss = train_one_epoch(model, tr_loader, optimizer, scaler)

    if epoch > CFG['warmup_epochs']:
        scheduler.step()

    va_auc = validate(model, va_loader)
    lr = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t_start
    log_rows.append(dict(epoch=epoch, lr=lr, tr_loss=tr_loss, va_auc=va_auc, time=elapsed))

    is_best = va_auc > best_auc
    print(f'Epoch {epoch:03d}/{CFG["epochs"]}'
          f' | LR={lr:.2e} | Loss={tr_loss:.4f} | AUC={va_auc:.4f}'
          f' | {elapsed:.0f}s{" <- best" if is_best else ""}')

    if is_best:
        best_auc, best_epoch = va_auc, epoch
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'best_auc': best_auc, 'labels': LABELS, 'cfg': CFG,
        }, WEIGHT_PATH_OUT)

    # Early stopping
    if epoch - best_epoch >= CFG['early_stopping']:
        print(f'Early stopping: no improvement for {CFG["early_stopping"]} epochs')
        break

    pd.DataFrame(log_rows).to_csv(f'{LOG_DIR}/train_log_phase3_fold{FOLD}.csv', index=False)

print('=' * 60)
print(f'Best AUC: {best_auc:.4f} @ Epoch {best_epoch}')
print('=' * 60)

In [ ]:
# ── Training Curve ────────────────────────────────────────
import matplotlib.pyplot as plt

log_df = pd.DataFrame(log_rows)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log_df['epoch'], log_df['tr_loss'])
axes[0].set_title('Train Loss (BCE)')
axes[0].set_xlabel('Epoch')
axes[1].plot(log_df['epoch'], log_df['va_auc'])
axes[1].set_title('Valid AUC (train_audio)')
axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/train_curve_phase3_fold{FOLD}.png', dpi=100)
plt.show()

total_time = sum(r['time'] for r in log_rows)
print(f'Total training time: {total_time/3600:.1f}h')
print(f'Best model: {WEIGHT_PATH_OUT}')
print()
print('Next steps:')
print('1. Save Version -> Save & Run All')
print('2. Output -> weights/best_fold0.pth -> New Dataset')
print('3. Update submission NB to use new weights')